# Obtaining Data from USDA Local Food Directory

In [8]:
import requests
import geopandas as gpd
from io import BytesIO 
import urllib.parse
import pandas as pd

In [7]:
# Obtaining shapefiles for target Texas counties 
# 1. Define the endpoint
base_url = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/1/query"

# 2. Build parameters (Note the addition of returnGeometry)
query_params = {
    "where": "BASENAME IN ('Dallas', 'Tarrant', 'Collin', 'Harris', 'Travis', 'Bexar') AND STATE='48'", 
    "outFields": "*",
    "returnGeometry": "true",  # CRITICAL: Forces the API to include the polygons
    "f": "geojson"
}

encoded_params = urllib.parse.urlencode(query_params)
request_url = f"{base_url}?{encoded_params}"

# 3. Fetch the data
response = requests.get(request_url)
response.raise_for_status()
data = response.json()

# 4. Ensure we actually got features back
if 'features' not in data or len(data['features']) == 0:
    raise ValueError("The API returned an empty dataset. Check your 'where' clause.")

# 5. Load into GeoPandas
target_counties = gpd.GeoDataFrame.from_features(data["features"])

# 6. CRITICAL FIX: Explicitly set the active geometry column
# We check to ensure the column exists first so it doesn't throw a confusing error
if 'geometry' in target_counties.columns:
    target_counties = target_counties.set_geometry('geometry')
else:
    raise KeyError("The API did not return a 'geometry' column. The API endpoint may not support GeoJSON export natively.")

# 7. Set the coordinate reference system
target_counties.set_crs(epsg=4326, inplace=True)

# Save it to disk
target_counties.to_file('texas_counties.geojson', driver='GeoJSON')
print("Successfully downloaded and saved spatial data.")

Successfully downloaded and saved spatial data.


In [ ]:
# Examining the type of response received from the USDA API
API_key = '4DBWYY5kkS'
response = requests.get(f'https://www.usdalocalfoodportal.com/api/csa/?apikey={API_key}&state=tx', headers=headers)
response.raise_for_status()

# Print the first 250 characters of the response
print(response.text[:250])

In [17]:
# 1. Load the county shapefile
counties = gpd.read_file('../fooddesertproject/texas_counties.geojson')

# 2. Make the API request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
API_key = '4DBWYY5kkS'
response = requests.get(f'https://www.usdalocalfoodportal.com/api/csa/?apikey={API_key}&state=tx', headers=headers)
response.raise_for_status()

# 2. Parse JSON into a standard Pandas DataFrame
data = response.json()

# If the list of points is nested under a key (e.g., data['results']), adjust this accordingly:
df = pd.DataFrame(data)

# 1. Unpack the nested JSON column
# .tolist() extracts the column of dictionaries into a standard Python list, 
# and passing that to pd.DataFrame() instantly expands the keys into individual columns.
expanded_df = pd.DataFrame(df['data'].tolist())

# 2. Convert the expanded DataFrame into a GeoDataFrame
raw_points = gpd.GeoDataFrame(
    expanded_df, 
    geometry=gpd.points_from_xy(expanded_df['location_x'], expanded_df['location_y']),
    crs="EPSG:4326"  # Standard API GPS coordinates
)

# 3. Load your target county boundaries
target_counties = gpd.read_file('texas_counties.geojson')

# 4. Clip the points (ensure CRS matches)
if raw_points.crs != target_counties.crs:
    raw_points = raw_points.to_crs(target_counties.crs)

clipped_points = gpd.clip(raw_points, target_county)

# 5. Save the final clipped data
clipped_points.to_file('usdaFarms.gpkg', driver='GPKG')

In [24]:
clipped_points['listing_name'].value_counts()

listing_name
Green Gate Farms                        2
Johnson's Backyard Garden               1
Tecolote Farm                           1
Munkebo Farm                            1
Plant It Forward                        1
Fisher Family Farm & Ranch              1
Booster Club Foods/eOrganic Products    1
GreenTexasFarms                         1
Squeezepenny Sustainable Farm           1
Name: count, dtype: int64

In [29]:
clipped_points.head(20)
clipped_points = clipped_points.drop([1])

In [30]:
clipped_points.head(20)

,directory_type,directory_name,updatetime,listing_image,listing_id,listing_name,listing_desc,brief_desc,mydesc,contact_name,...,location_address,location_state,location_city,location_street,location_zipcode,location_x,location_y,term,distance,geometry
25,csa,csa enterprise,"Sep 15th, 2014",default-csa-4-3.jpg,200341,Johnson's Backyard Garden,None,Open: Year-round<br>Available Products: Gooseb...,Open: Year-round<br>Available Products: Gooseb...,Brenton Johnson or Carrie Kenny,...,"9515 Hergotz Lane, Box E, Austin, Texas 78742",Texas,Austin,9515 Hergotz Ln,78742,-97.6547686,30.2369381,,1000,POINT (-97.65477 30.23694)
19,csa,csa enterprise,"Mar 23rd, 2015",default-csa-4-3.jpg,200343,Tecolote Farm,None,Open: March to December<br>Available Products:...,Open: March to December<br>Available Products:...,David Pitre and Katie Kraemer,...,"16301 Decker Lake Rd, Manor, Texas 78653",Texas,Manor,16301 Decker Lake Rd,78653,-97.5627895,30.2606663,,1000,POINT (-97.56279 30.26067)
23,csa,csa enterprise,"Nov 7th, 2017",default-csa-4-3.jpg,200760,Green Gate Farms,None,Open: February to December<br>Available Produc...,Open: February to December<br>Available Produc...,Erin Flynn,...,"8310 Canoga Ave, Austin , Texas 78724",Texas,Austin,8310 Canoga Ave,78724,-97.6348801,30.2857477,,1000,POINT (-97.63488 30.28575)
3,csa,csa enterprise,"Aug 12th, 2014",default-csa-4-3.jpg,200437,Munkebo Farm,None,Open: Year-round<br>Available Products: Artich...,Open: Year-round<br>Available Products: Artich...,Germaine Swenson,...,"20826, Manor, Texas 78653",Texas,Manor,,78653,-97.5569456,30.3407629,,1000,POINT (-97.55695 30.34076)
4,csa,csa enterprise,"Aug 13th, 2019",default-csa-4-3.jpg,200838,Plant It Forward,None,Open: Year-round<br>Available Products: Avocad...,Open: Year-round<br>Available Products: Avocad...,Daniella Lewis,...,"4030 Willowbend Blvd, Houston, Texas 77025",Texas,Houston,4030 Willowbend Blvd,77025,-95.4433233,29.6650135,,1000,POINT (-95.44332 29.66501)
22,csa,csa enterprise,"Nov 11th, 2014",default-csa-4-3.jpg,200340,Fisher Family Farm & Ranch,None,Open: April to November<br>Available Products:...,Open: April to November<br>Available Products:...,David Fisher,...,", Dallas , Texas 75220",Texas,Dallas,,75220,-96.8726295,32.8621631,,1000,POINT (-96.87263 32.86216)
2,csa,csa enterprise,"Aug 11th, 2017",default-csa-4-3.jpg,200754,Booster Club Foods/eOrganic Products,None,Open: Year-round<br>Available Products: Salmon...,Open: Year-round<br>Available Products: Salmon...,Rhod Williams,...,"2591 Dallas Parkway #103, Frisco, Texas 75034",Texas,Frisco,2591 Dallas Pkwy,75034,-96.8251348,33.0997558,,1000,POINT (-96.82513 33.09976)
20,csa,csa enterprise,"Mar 31st, 2022",default-csa-4-3.jpg,200844,GreenTexasFarms,,Open: Year-round<br>Available Products: Curran...,Open: Year-round<br>Available Products: Curran...,GreenTexasFarms,...,"296 W FOREST GROVE RD, Lucas, Texas 75002",Texas,Lucas,296 W Forest Grove Rd,75002,-96.579715,33.1207822,,1000,POINT (-96.57972 33.12078)
24,csa,csa enterprise,"Oct 2nd, 2014",default-csa-4-3.jpg,200551,Squeezepenny Sustainable Farm,None,Open: April to November<br>Available Products:...,Open: April to November<br>Available Products:...,Penny Braley,...,"3723 County Road 412, McKinney, Texas 75071",Texas,McKinney,3723 County Rd 412,75071,-96.541131,33.2600259,,1000,POINT (-96.54113 33.26003)
